In [1]:
import inspect
import json
import os
from dataclasses import dataclass
from pathlib import Path

import torch
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

In [2]:
# Cell 2: 训练配置
DATA_DIR = Path("/content/drive/MyDrive/nlpcc/data")
OUT_DIR = Path("/content/drive/MyDrive/nlpcc/output/track2/sft")
OUT_DIR.mkdir(parents = True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "track2" /"sft_train.jsonl"
DEV_FILE = DATA_DIR / "track2" /"sft_dev.jsonl"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # 可以替换为本地模型路径或其他开源模型
MAX_LENGTH = 1024
MAX_PROMPT_LENGTH = 768
LR = 2e-5
EPOCHS = 3
TRAIN_BS = 2
EVAL_BS = 2
GRAD_ACCUM = 8
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.0

USE_LORA = True
LOAD_IN_4BIT = False
BF16 = True
FP16 = False


In [3]:
# Cell 3: 数据读取

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


train_rows = read_jsonl(TRAIN_FILE)
dev_rows = read_jsonl(DEV_FILE)

print("train:", len(train_rows))
print("dev  :", len(dev_rows))
print("\nPrompt sample:\n", train_rows[0]["prompt"][:800])
print("\nResponse sample:\n", train_rows[0]["response"])

train: 3520
dev  : 514

Prompt sample:
 You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
You work in a corporate setting where your manager frequently imposes strict guidelines.

Question:
How would you handle a disagreement with a superior during a team meeting?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:


Response sample:
 I would suggest alternative solutions diplomatically, ensuring my input is heard without challenging the manager openly, prioritizing team harmony and respect for hierarchy.


In [4]:
# Cell 4: 加载 tokenizer 与 causal LM
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code = True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
if tokenizer.pad_token is None:
  tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model_kwargs = {"trust_remote_code": True}
if LOAD_IN_4BIT:
  from transformers import BitsAndBytesConfig
  model_kwargs["quantization_config"] = BitsAndBytesConfig(
      load_in_4bit = True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.bfloat16 if BF16 else torch.float16,
  )
  model_kwargs["device_map"] = "auto"
elif BF16:
  model_kwargs["torch_dtype"] = torch.bfloat16
elif FP16:
  model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

if not LOAD_IN_4BIT:
  model = model.cuda()

print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("model dtype:", next(model.parameters()).dtype)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

pad token: <|endoftext|> 151643
model dtype: torch.bfloat16


In [5]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [6]:
# Cell 5: LoRA 配置
if USE_LORA:
  from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
  if LOAD_IN_4BIT:
    model = prepare_model_for_kbit_training(model)

  lora_config = LoraConfig(
      r = 16,
      lora_alpha = 32,
      lora_dropout=0.05,
      bias = "none",
      task_type = "CAUSAL_LM",
      target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],

  )
  model = get_peft_model(model, lora_config)
  model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,561,762,816 || trainable%: 1.1823


In [7]:
#cell 6: Dataset构造
class SFTDataset(Dataset):
  def __init__(self, rows, tokenizer, max_length = 1024, max_prompt_length = 768):
    self.examples = []
    eos = tokenizer.eos_token or ""
    for row in rows:
      prompt_ids = tokenizer(
          row["prompt"],
          add_special_tokens = True,
          truncation = True,
          max_length = max_prompt_length,
          )["input_ids"]
      remaining = max(1, max_length - len(prompt_ids))
      response_ids = tokenizer(
          row["response"] + eos,
          add_special_tokens = False,
          truncation = True,
          max_length = remaining,
      )["input_ids"]

      input_ids = prompt_ids + response_ids
      labels = [-100]*len(prompt_ids) + response_ids
      self.examples.append({
          "input_ids":input_ids,
          "attention_mask":[1]*len(input_ids),
          "labels":labels,}
      )
  def __len__(self):
    return len(self.examples)
  def __getitem__(self, idx):
    return self.examples[idx]

class SFTCollator:
  def __init__(self, pad_token_id):
    self.pad_token_id = pad_token_id
  def __call__(self, features):
    max_len = max(len(x["input_ids"]) for x in features)
    batch = {"input_ids":[], "attention_mask":[], "labels":[]}
    for x in features:
      pad_len = max_len - len(x["input_ids"])
      batch["input_ids"].append(x["input_ids"] + [self.pad_token_id]*pad_len)
      batch["attention_mask"].append(x["attention_mask"] + [0]*pad_len)
      batch["labels"].append(x["labels"] + [-100]*pad_len)
    return {k:torch.tensor(v, dtype = torch.long) for k, v in batch.items()}

train_dataset = SFTDataset(train_rows, tokenizer, MAX_LENGTH, MAX_PROMPT_LENGTH)
dev_dataset = SFTDataset(dev_rows, tokenizer, MAX_LENGTH, MAX_PROMPT_LENGTH)

print("train dataset:", len(train_dataset))
print("dev dataset  :", len(dev_dataset))
print("first input length:", len(train_dataset[0]["input_ids"]))

train dataset: 3520
dev dataset  : 514
first input length: 125


In [8]:
# Cell 7: TrainingArguments 与 Trainer

training_kwargs = dict(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=10,
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    bf16=BF16,
    fp16=FP16,
    report_to="none",
    remove_unused_columns=False,
)
if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_kwargs["eval_strategy"] = "steps"
else:
    training_kwargs["evaluation_strategy"] = "steps"

training_args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=SFTCollator(tokenizer.pad_token_id),
)

trainer

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [9]:
trainer.train()
trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))

Step,Training Loss,Validation Loss
100,1.811873,1.789308
200,1.705395,1.707422
300,1.651137,1.671445
400,1.581935,1.647481
500,1.561395,1.633855
600,1.511085,1.628554


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetunin

('/content/drive/MyDrive/nlpcc/output/track2/sft/tokenizer_config.json',
 '/content/drive/MyDrive/nlpcc/output/track2/sft/chat_template.jinja',
 '/content/drive/MyDrive/nlpcc/output/track2/sft/tokenizer.json')

In [10]:
def generate_response(prompt, max_new_tokens=96):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    gen_ids = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


print("Prompt:\n", dev_rows[0]["prompt"])
print("Gold:\n", dev_rows[0]["response"])
print("Generation:\n", generate_response(dev_rows[0]["prompt"]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prompt:
 You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
A teammate insists on frequent communication to feel secure.

Question:
How would you handle the teammate’s demand for constant updates?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:

Gold:
 I would prioritize maintaining that frequent contact despite my schedule, as I want to ensure my teammate feels valued, included, and that our teamwork remains perfectly smooth.
Generation:
 I would prioritize maintaining harmony by limiting my own requests, ensuring I don’t cause them distress or make them feel uncomfortable.
